## Part 1. Test Code

Reference: https://scikit-image.org/docs/stable/auto_examples/edges/plot_active_contours.html

Active contours (snakes) are energy-minimizing splines guided by external image forces
and internal smoothness constraints. The circle below is the initial curve; it deforms
to fit the astronaut's head.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.color import rgb2gray
from skimage import data
from skimage.filters import gaussian
from skimage.segmentation import active_contour

img = data.astronaut()
img = rgb2gray(img)

s = np.linspace(0, 2 * np.pi, 400)
r = 100 + 100 * np.sin(s)
c = 220 + 100 * np.cos(s)
init = np.array([r, c]).T

snake = active_contour(
    gaussian(img, sigma=3, preserve_range=False),
    init,
    alpha=0.015,
    beta=10,
    gamma=0.001,
)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img, cmap=plt.cm.gray)
ax.plot(init[:, 1], init[:, 0], '--r', lw=3)
ax.plot(snake[:, 1], snake[:, 0], '-b', lw=3)
ax.set_xticks([]), ax.set_yticks([])
ax.axis([0, img.shape[1], img.shape[0], 0])

plt.show()

## Part 2. Person Segmentation

Given a bounding box annotation (YOLO format) of a person in a drone image, we:
1. Convert the bounding box to a rectangular initial curve
2. Run active contours (snake) to find the person's boundaries
3. Create a binary segmentation mask from the snake
4. Display the bounding box, snake, and blended mask over the original image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.color import rgb2gray
from skimage import data
from skimage.filters import gaussian
from skimage.segmentation import active_contour
import matplotlib.image as mpimg
from skimage import draw

In [ ]:
def yolo2xyxy(x: float, y: float, w: float, h: float, img_size: tuple) -> list:
  '''
    Transform YOLO bounding box format annotations to xyxy format.
    YOLO format: see https://docs.ultralytics.com/datasets/detect/#ultralytics-yolo-format
    xyxy format: (x1,y1) -> top-left coordinate, (x2,y2) -> bottom-right coordinate

  '''
  img_h, img_w = img_size[0], img_size[1]

  x1 = int((x - w / 2) * img_w)
  y1 = int((y - h / 2) * img_h)
  x2 = int((x + w / 2) * img_w)
  y2 = int((y + h / 2) * img_h)

  return [x1, y1, x2, y2]


def get_curve_from_bbox(x1: int, y1: int, x2: int, y2: int) -> np.array:
  ''' Output is a [r,c] array of points describing a parametric curve. '''

  n_per_side = 100
  curve = []

  # Top edge: row=y1, col varies from x1 to x2
  for c in np.linspace(x1, x2, n_per_side):
    curve.append([y1, c])
  # Right edge: col=x2, row varies from y1 to y2
  for r in np.linspace(y1, y2, n_per_side):
    curve.append([r, x2])
  # Bottom edge: row=y2, col varies from x2 to x1
  for c in np.linspace(x2, x1, n_per_side):
    curve.append([y2, c])
  # Left edge: col=x1, row varies from y2 to y1
  for r in np.linspace(y2, y1, n_per_side):
    curve.append([r, x1])

  return np.array(curve)


In [ ]:
# Reference source code: https://scikit-image.org/docs/stable/auto_examples/edges/plot_active_contours.html

# 1. Load the frame and annotations
frame = "Actor031_a10_f0001"
img_color = mpimg.imread(f"{frame}.jpg")
annotation = np.loadtxt(f"{frame}.txt")[1:].tolist()

# 1.5 Adapt format of image and annotations
img_gray = rgb2gray(img_color)
xyxy = yolo2xyxy(*annotation, img_gray.shape)
print("Bounding box (x1,y1,x2,y2):", xyxy)

# 2. Create your initial curve
init = get_curve_from_bbox(*xyxy)

# 3. Run snake function
### NOTE: Feel free to play with parameters here ###
snake = active_contour(
    gaussian(img_gray, sigma=3, preserve_range=False),
    init,
    alpha=0.015,
    beta=10,
    gamma=0.001,
)

In [ ]:
# 4. Create segmentation mask from snake boundaries
mask = np.zeros(img_gray.shape, dtype=np.uint8)
fill_row_coords, fill_col_coords = draw.polygon(snake[:, 0], snake[:, 1], img_gray.shape)
mask[fill_row_coords, fill_col_coords] = 1

plt.imshow(mask, cmap='gray')
plt.title("Segmentation Mask")
plt.show()

In [ ]:
# 5. Display snake, bbox, and overlayed segmentation mask on color image
color = True # Change this to switch between color and grayscale display
if color:
  img = img_color
  img_h, img_w, _ = img.shape
else:
  img = img_gray
  img_h, img_w = img.shape

dpi=600
fig, ax = plt.subplots(figsize=( img_w/dpi, img_h/dpi), dpi=dpi)
ax = fig.add_axes([0, 0, 1, 1])
ax.imshow(img, cmap=plt.cm.gray)
masked_data = np.ma.masked_where(mask == 0, mask)
ax.imshow(masked_data, cmap='jet', alpha=0.5)
ax.plot(init[:, 1], init[:, 0], '--r', lw=1)
ax.plot(snake[:, 1], snake[:, 0], '-b', lw=1)
ax.set_xlim(0, img_w)
ax.set_ylim(img_h, 0)
ax.axis('off')

plt.savefig('img_results.png', dpi=dpi, pad_inches=0)

plt.show()

## Reflections on Active Contours

Active contours (snakes) are deformable curves that minimize an energy function
combining **internal energy** (smoothness/elasticity) and **external energy** (image
gradient forces). The three key parameters are:

- **alpha** (elasticity): controls how much the snake stretches. Higher values keep the
  curve compact and resist stretching.
- **beta** (stiffness/rigidity): controls resistance to bending. Higher values produce
  smoother, more rigid curves.
- **gamma** (step size): controls the time step of the gradient descent optimization.

**Observations from this task:**

The snake was initialized as a rectangle tightly matching the ground-truth bounding box.
It converged well to the person's silhouette in the aerial drone image. However, a few
challenges were noted:

1. **Sensitivity to initialization**: Active contours are local optimizers — they only
   work if the initial curve is already near the target boundary. Without the YOLO
   bounding box, placing the initial curve correctly would be very difficult.

2. **Background complexity**: The green, textured foliage creates many competing edges.
   The Gaussian smoothing (sigma=3) helps suppress noise but also blurs the person's
   edges slightly.

3. **Scale sensitivity**: The person occupies a small region (~117×274 pixels) in a
   5472×3078 image. The snake parameters tuned for the full astronaut image (which
   fills the frame) may need adjustment for small-scale targets.

4. **No topology changes**: Snakes cannot split or merge, limiting their use to
   single connected regions. Modern level-set methods overcome this.

Overall, active contours are an elegant classical segmentation technique that remains
useful when a reasonable initialization is available (e.g., from a detector bounding box).